<a href="https://colab.research.google.com/github/hideaki-kyutech/syseng2026/blob/main/week9_PDcontrol_GA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### プロット図の日本語化対応（必要なら）

In [13]:
!apt-get -y install fonts-ipafont-gothic
!pip install japanize-matplotlib
import japanize_matplotlib
import matplotlib.pyplot as plt

# 場合によっては必要
plt.rcParams['font.family'] = 'IPAPGothic'
japanize_matplotlib.japanize()

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  fonts-ipafont-mincho
The following NEW packages will be installed:
  fonts-ipafont-gothic fonts-ipafont-mincho
0 upgraded, 2 newly installed, 0 to remove and 3 not upgraded.
Need to get 8,237 kB of archives.
After this operation, 28.7 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 fonts-ipafont-gothic all 00303-21ubuntu1 [3,513 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 fonts-ipafont-mincho all 00303-21ubuntu1 [4,724 kB]
Fetched 8,237 kB in 0s (34.8 MB/s)
Selecting previously unselected package fonts-ipafont-gothic.
(Reading database ... 118252 files and directories currently installed.)
Preparing to unpack .../fonts-ipafont-gothic_00303-21ubuntu1_all.deb ...
Unpacking fonts-ipafont-gothic (00303-21ubuntu1) ...
Selecting previously unselected package fonts-ipafo

### 演習1: 手作業によるPDゲイン調整
$K_p$と$K_d$を変更し、scoreをできるだけ高くする値を探してください。

In [ ]:
# ============================================
# 第9回 演習1：
# 手作業によるPDゲイン調整
# 現実化モデル：
# 入力遅延・入力レート制限・センサノイズあり
# ============================================

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

np.random.seed(0)

# ============================================
# 1. シミュレーション設定【変更不可】
# ============================================

dt = 0.1
T = 30
steps = int(T / dt)
time = np.arange(steps) * dt

d_target = 20.0

# 車両モデル
c = 0.08
u_min = -7.0
u_max = 3.0

# 現実化要素
delay_sec = 0.8
delay_steps = int(delay_sec / dt)

du_max = 0.25

distance_noise_std = 0.3
velocity_noise_std = 0.2

# 初期状態
d0 = 20.0
v0 = 15.0

EVAL_SEED = 0


def front_velocity(t):
    """
    前方車両速度 [m/s]
    0〜8秒   : 15 m/s
    8〜16秒  : 8 m/s  急減速
    16秒以降 : 15 m/s 回復
    """
    if t < 8:
        return 15.0
    elif t < 16:
        return 8.0
    else:
        return 15.0


v_front_profile = np.array([front_velocity(t) for t in time])


# ============================================
# 2. 学生が調整するPDゲイン【変更可能】
# ============================================

# 初期状態は無制御状態
K_p = 0.0
K_d = 0.0

# ============================================
# 3. 評価関数の重み【変更不可】
# ============================================

weights = {
    "error": 1.0,    # 平均絶対距離誤差
    "risk": 8.0,     # 最大接近超過
    "far": 0.5,      # 最大離れすぎ
    "energy": 0.5,   # 制御入力エネルギー
    "smooth": 10.0   # 制御入力変化量
}


# ============================================
# 4. シミュレーション関数【変更不可】
# ============================================

def simulate_pd_realistic(K_p, K_d, seed=EVAL_SEED):
    np.random.seed(seed)

    d = np.zeros(steps)
    v = np.zeros(steps)

    measured_d = np.zeros(steps)
    measured_v = np.zeros(steps)

    u_cmd = np.zeros(steps)
    u_delay = np.zeros(steps)
    u_applied = np.zeros(steps)

    e_true = np.zeros(steps)
    e_meas = np.zeros(steps)

    d[0] = d0
    v[0] = v0

    for k in range(steps - 1):
        vf = front_velocity(time[k])

        # 真の距離誤差
        e_true[k] = d[k] - d_target

        # センサノイズを含む測定値
        measured_d[k] = d[k] + np.random.normal(0, distance_noise_std)
        measured_v[k] = v[k] + np.random.normal(0, velocity_noise_std)

        # 測定値に基づく誤差
        e_meas[k] = measured_d[k] - d_target

        # 相対速度：距離誤差の変化速度に相当
        de_meas = vf - measured_v[k]

        # PD制御入力
        u_cmd[k] = K_p * e_meas[k] + K_d * de_meas
        u_cmd[k] = np.clip(u_cmd[k], u_min, u_max)

        # 制御入力遅延
        if k - delay_steps >= 0:
            u_delay[k] = u_cmd[k - delay_steps]
        else:
            u_delay[k] = 0.0

        # 入力レート制限
        if k == 0:
            u_applied[k] = u_delay[k]
        else:
            u_applied[k] = np.clip(
                u_delay[k],
                u_applied[k-1] - du_max,
                u_applied[k-1] + du_max
            )

        # 車両の離散時間モデル
        v[k+1] = v[k] + dt * (u_applied[k] - c * v[k])
        v[k+1] = max(v[k+1], 0.0)

        d[k+1] = d[k] + dt * (vf - v[k])
        d[k+1] = max(d[k+1], 0.0)

    e_true[-1] = d[-1] - d_target
    measured_d[-1] = d[-1]
    measured_v[-1] = v[-1]
    e_meas[-1] = measured_d[-1] - d_target
    u_cmd[-1] = u_cmd[-2]
    u_delay[-1] = u_delay[-2]
    u_applied[-1] = u_applied[-2]

    return {
        "d": d,
        "v": v,
        "measured_d": measured_d,
        "measured_v": measured_v,
        "u_cmd": u_cmd,
        "u_delay": u_delay,
        "u_applied": u_applied,
        "e_true": e_true,
        "e_meas": e_meas
    }


# ============================================
# 5. 評価指標とscore計算【変更不可】
# ============================================

def evaluate_result(result):
    d = result["d"]
    u = result["u_applied"]
    err = d - d_target

    mean_abs_error = np.mean(np.abs(err))
    close_risk = abs(min(np.min(err), 0.0))
    too_far = max(np.max(err), 0.0)
    min_distance = np.min(d)
    control_energy = np.mean(u**2)
    control_smoothness = np.mean(np.abs(np.diff(u)))

    cost = (
        weights["error"]  * mean_abs_error +
        weights["risk"]   * close_risk +
        weights["far"]    * too_far +
        weights["energy"] * control_energy +
        weights["smooth"] * control_smoothness
    )

    score = 100 - cost

    return {
        "平均絶対距離誤差": mean_abs_error,
        "最大接近超過": close_risk,
        "最大離れすぎ": too_far,
        "最小車間距離": min_distance,
        "制御入力エネルギー": control_energy,
        "制御入力変化量": control_smoothness,
        "cost": cost,
        "score": score
    }


# ============================================
# 6. 実行
# ============================================

result = simulate_pd_realistic(K_p, K_d)
score_info = evaluate_result(result)

print("=== 現在のPDゲイン ===")
print(f"K_p = {K_p}")
print(f"K_d = {K_d}")

print("\n=== 現実化モデルの条件 ===")
print(f"制御入力遅延: {delay_sec} 秒")
print(f"入力変化上限 du_max: {du_max}")
print(f"距離ノイズ標準偏差: {distance_noise_std} m")
print(f"速度ノイズ標準偏差: {velocity_noise_std} m/s")

print("\n=== 評価指標 ===")
for k, val in score_info.items():
    print(f"{k}: {val:.3f}")

trial_result = pd.DataFrame([{
    "K_p": K_p,
    "K_d": K_d,
    **score_info
}])

display(trial_result)


# ============================================
# 7. 結果表示
# ============================================

plt.figure(figsize=(11, 5))
plt.plot(time, v_front_profile, linestyle="--", label="Front vehicle")
plt.plot(time, result["v"], label="Own vehicle")
plt.xlabel("Time [s]")
plt.ylabel("Velocity [m/s]")
plt.title("Vehicle Velocity")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(11, 5))
plt.plot(time, result["d"], label="True distance")
plt.plot(time, result["measured_d"], alpha=0.4, label="Measured distance")
plt.axhline(d_target, linestyle="--", label="Target distance")
plt.xlabel("Time [s]")
plt.ylabel("Distance [m]")
plt.title("Inter-vehicle Distance")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(11, 5))
plt.plot(time, result["e_true"], label="True distance error")
plt.plot(time, result["e_meas"], alpha=0.4, label="Measured distance error")
plt.axhline(0, linestyle="--")
plt.xlabel("Time [s]")
plt.ylabel("Distance Error [m]")
plt.title("Distance Error")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(11, 5))
plt.plot(time, result["u_cmd"], label="Commanded input")
plt.plot(time, result["u_delay"], label="Delayed input", linestyle="--")
plt.plot(time, result["u_applied"], label="Applied input", linewidth=2)
plt.xlabel("Time [s]")
plt.ylabel("Control Input")
plt.title("Control Input: Command, Delay, Rate Limit")
plt.legend()
plt.grid(True)
plt.show()


# ============================================
# 8. 複数ゲインの比較【参考】
# ============================================

test_gains = [
    (0.0, 0.0),
    (2.0, 0.0),
    (0.0, 2.0),
    (2.0, 2.0)
    ]

rows = []

plt.figure(figsize=(11, 5))

for kp, kd in test_gains:
    res = simulate_pd_realistic(kp, kd)
    info = evaluate_result(res)

    rows.append({
        "K_p": kp,
        "K_d": kd,
        **info
    })

    plt.plot(time, res["d"], label=f"Kp={kp}, Kd={kd}")

plt.axhline(d_target, linestyle="--", label="Target distance")
plt.xlabel("Time [s]")
plt.ylabel("Distance [m]")
plt.title("Distance Response for Several Gains")
plt.legend()
plt.grid(True)
plt.show()

comparison_df = pd.DataFrame(rows)
display(comparison_df)

### 演習2: GAによるPD制御ゲイン最適化

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# =========================================================
# 【変更不可】共通シミュレーション条件
# =========================================================

dt = 0.1
T = 30
steps = int(T / dt)
time = np.arange(steps) * dt

d_target = 20.0

# 車両モデル
c = 0.08
u_min = -7.0
u_max = 3.0

# 現実化要素
delay_sec = 0.8
delay_steps = int(delay_sec / dt)

du_max = 0.25

distance_noise_std = 0.3
velocity_noise_std = 0.2

# 初期状態
d0 = 20.0
v0 = 15.0

# 乱数固定：全員同じ条件で比較
EVAL_SEED = 0


def front_velocity(t):
    if t < 8:
        return 15.0
    elif t < 16:
        return 8.0
    else:
        return 15.0


v_front_profile = np.array([front_velocity(t) for t in time])


# =========================================================
# 【変更不可】車両シミュレーション
# =========================================================

def simulate_pd_realistic(K_p, K_d, seed=EVAL_SEED):
    np.random.seed(seed)

    d = np.zeros(steps)
    v = np.zeros(steps)

    measured_d = np.zeros(steps)
    measured_v = np.zeros(steps)

    u_cmd = np.zeros(steps)
    u_delay = np.zeros(steps)
    u_applied = np.zeros(steps)

    e_true = np.zeros(steps)
    e_meas = np.zeros(steps)

    d[0] = d0
    v[0] = v0

    for k in range(steps - 1):
        vf = front_velocity(time[k])

        e_true[k] = d[k] - d_target

        measured_d[k] = d[k] + np.random.normal(0, distance_noise_std)
        measured_v[k] = v[k] + np.random.normal(0, velocity_noise_std)

        e_meas[k] = measured_d[k] - d_target
        de_meas = vf - measured_v[k]

        u_cmd[k] = K_p * e_meas[k] + K_d * de_meas
        u_cmd[k] = np.clip(u_cmd[k], u_min, u_max)

        if k - delay_steps >= 0:
            u_delay[k] = u_cmd[k - delay_steps]
        else:
            u_delay[k] = 0.0

        if k == 0:
            u_applied[k] = u_delay[k]
        else:
            u_applied[k] = np.clip(
                u_delay[k],
                u_applied[k-1] - du_max,
                u_applied[k-1] + du_max
            )

        v[k+1] = v[k] + dt * (u_applied[k] - c * v[k])
        v[k+1] = max(v[k+1], 0.0)

        d[k+1] = d[k] + dt * (vf - v[k])
        d[k+1] = max(d[k+1], 0.0)

    e_true[-1] = d[-1] - d_target
    measured_d[-1] = d[-1]
    measured_v[-1] = v[-1]
    e_meas[-1] = measured_d[-1] - d_target
    u_cmd[-1] = u_cmd[-2]
    u_delay[-1] = u_delay[-2]
    u_applied[-1] = u_applied[-2]

    return {
        "d": d,
        "v": v,
        "measured_d": measured_d,
        "measured_v": measured_v,
        "u_cmd": u_cmd,
        "u_delay": u_delay,
        "u_applied": u_applied,
        "e_true": e_true,
        "e_meas": e_meas
    }


# =========================================================
# 【変更不可】共通評価指標
# =========================================================

def calculate_metrics(result):
    d = result["d"]
    u = result["u_applied"]
    err = d - d_target

    return {
        "平均絶対距離誤差": np.mean(np.abs(err)),
        "最大接近超過": abs(min(np.min(err), 0.0)),
        "最大離れすぎ": max(np.max(err), 0.0),
        "最小車間距離": np.min(d),
        "制御入力エネルギー": np.mean(u**2),
        "制御入力変化量": np.mean(np.abs(np.diff(u)))
    }


# =========================================================
# 【変更不可】コンペ用スコア
# 小さい cost ほど良い
# score は大きいほど良い
# =========================================================

COMPETITION_WEIGHTS = {
    "error": 1.0,
    "risk": 8.0,
    "far": 0.5,
    "energy": 0.5,
    "smooth": 10.0
}


def evaluate_controller(K_p, K_d):
    result = simulate_pd_realistic(K_p, K_d, seed=EVAL_SEED)
    m = calculate_metrics(result)

    cost = (
        COMPETITION_WEIGHTS["error"]  * m["平均絶対距離誤差"] +
        COMPETITION_WEIGHTS["risk"]   * m["最大接近超過"] +
        COMPETITION_WEIGHTS["far"]    * m["最大離れすぎ"] +
        COMPETITION_WEIGHTS["energy"] * m["制御入力エネルギー"] +
        COMPETITION_WEIGHTS["smooth"] * m["制御入力変化量"]
    )

    score = 100 - cost

    info = {
        "K_p": K_p,
        "K_d": K_d,
        **m,
        "cost": cost,
        "score": score
    }

    return score, info, result


# =========================================================
# 【ここから変更可能】
# 学生が調整してよい部分
# =========================================================

STUDENT_NAME = "Your Name"

# GA設定
pop_size = 10
generations = 10

# 計算回数制限
# pop_size × generations が 1500 以下になるようにする
MAX_EVALUATIONS = 1500

# 探索範囲
Kp_min, Kp_max = 0.0, 100.0
Kd_min, Kd_max = 0.0, 100.0

# GAパラメータ
crossover_rate = 0.3
mutation_rate = 0.3
mutation_scale = 1.00

# 乱数シード
# 変更して複数回試してもよい
GA_SEED = 1


# =========================================================
# 【変更可能】GA部品
# ただし、まずはそのまま実行してよい
# =========================================================

def create_individual():
    return np.array([
        np.random.uniform(Kp_min, Kp_max),
        np.random.uniform(Kd_min, Kd_max)
    ])


def create_population():
    return np.array([create_individual() for _ in range(pop_size)])


def roulette_selection(population, fitnesses):

    # fitnessが負になる場合があるのでシフト

    shifted = fitnesses - np.min(fitnesses) + 1e-6

    probs = shifted / np.sum(shifted)

    idx = np.random.choice(

        len(population),

        p=probs

    )

    return population[idx].copy()


def crossover(parent1, parent2):
    if np.random.rand() < crossover_rate:
        alpha = np.random.rand()
        child1 = alpha * parent1 + (1 - alpha) * parent2
        child2 = alpha * parent2 + (1 - alpha) * parent1
        return child1, child2
    else:
        return parent1.copy(), parent2.copy()


def mutate(individual):
    if np.random.rand() < mutation_rate:
        individual += np.random.normal(0, mutation_scale, size=2)

    individual[0] = np.clip(individual[0], Kp_min, Kp_max)
    individual[1] = np.clip(individual[1], Kd_min, Kd_max)

    return individual


# =========================================================
# 【変更不可】設定チェック
# =========================================================

if pop_size * generations > MAX_EVALUATIONS:
    raise ValueError(
        f"計算回数が多すぎます。pop_size × generations = {pop_size * generations} です。"
        f"{MAX_EVALUATIONS} 以下にしてください。"
    )


# =========================================================
# 【変更不可】GA実行
# =========================================================

np.random.seed(GA_SEED)

population = create_population()

history = []
best_individual = None
best_score = -np.inf
best_info = None
best_result = None

for gen in range(generations):

    eval_results = [
        evaluate_controller(ind[0], ind[1])
        for ind in population
    ]

    scores = np.array([r[0] for r in eval_results])
    infos = [r[1] for r in eval_results]
    sim_results = [r[2] for r in eval_results]

    gen_best_idx = np.argmax(scores)

    if scores[gen_best_idx] > best_score:
        best_score = scores[gen_best_idx]
        best_individual = population[gen_best_idx].copy()
        best_info = infos[gen_best_idx]
        best_result = sim_results[gen_best_idx]

    history.append({
        "generation": gen,
        "best_score": np.max(scores),
        "mean_score": np.mean(scores),
        "best_Kp": population[gen_best_idx][0],
        "best_Kd": population[gen_best_idx][1]
    })

    new_population = []

    # エリート保存
    new_population.append(population[gen_best_idx].copy())

    while len(new_population) < pop_size:
        p1 = roulette_selection(population, scores)
        p2 = roulette_selection(population, scores)

        c1, c2 = crossover(p1, p2)

        c1 = mutate(c1)
        c2 = mutate(c2)

        new_population.append(c1)

        if len(new_population) < pop_size:
            new_population.append(c2)

    population = np.array(new_population)

history_df = pd.DataFrame(history)


# =========================================================
# 【変更不可】結果表示
# =========================================================

print("=== コンペ結果 ===")
print(f"Student: {STUDENT_NAME}")
print(f"Evaluations: {pop_size * generations}")
print(f"Best K_p: {best_info['K_p']:.4f}")
print(f"Best K_d: {best_info['K_d']:.4f}")
print(f"Best Score: {best_info['score']:.4f}")
print(f"Cost: {best_info['cost']:.4f}")

print("\n=== 評価指標 ===")
for key in [
    "平均絶対距離誤差",
    "最大接近超過",
    "最大離れすぎ",
    "最小車間距離",
    "制御入力エネルギー",
    "制御入力変化量"
]:
    print(f"{key}: {best_info[key]:.4f}")


summary_df = pd.DataFrame([{
    "student": STUDENT_NAME,
    "evaluations": pop_size * generations,
    "K_p": best_info["K_p"],
    "K_d": best_info["K_d"],
    "score": best_info["score"],
    "cost": best_info["cost"],
    "平均絶対距離誤差": best_info["平均絶対距離誤差"],
    "最大接近超過": best_info["最大接近超過"],
    "最大離れすぎ": best_info["最大離れすぎ"],
    "最小車間距離": best_info["最小車間距離"],
    "制御入力エネルギー": best_info["制御入力エネルギー"],
    "制御入力変化量": best_info["制御入力変化量"]
}])

display(summary_df)


# =========================================================
# 【変更不可】GA探索過程の可視化
# =========================================================

plt.figure(figsize=(10, 5))
plt.plot(history_df["generation"], history_df["best_score"], label="Best score")
plt.plot(history_df["generation"], history_df["mean_score"], label="Mean score")
plt.xlabel("Generation")
plt.ylabel("Score")
plt.title("GA Optimization Process")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(history_df["generation"], history_df["best_Kp"], label="Best Kp")
plt.plot(history_df["generation"], history_df["best_Kd"], label="Best Kd")
plt.xlabel("Generation")
plt.ylabel("Gain value")
plt.title("Evolution of PD Gains")
plt.legend()
plt.grid(True)
plt.show()


# =========================================================
# 【変更不可】制御結果の可視化
# =========================================================

plt.figure(figsize=(11, 5))
plt.plot(time, v_front_profile, linestyle="--", label="Front vehicle")
plt.plot(time, best_result["v"], label="Own vehicle")
plt.xlabel("Time [s]")
plt.ylabel("Velocity [m/s]")
plt.title("Vehicle Velocity: GA Optimized PD")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(11, 5))
plt.plot(time, best_result["d"], label="Distance")
plt.axhline(d_target, linestyle="--", label="Target distance")
plt.xlabel("Time [s]")
plt.ylabel("Distance [m]")
plt.title("Inter-vehicle Distance: GA Optimized PD")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(11, 5))
plt.plot(time, best_result["e_true"], label="Distance error")
plt.axhline(0, linestyle="--")
plt.xlabel("Time [s]")
plt.ylabel("Distance Error [m]")
plt.title("Distance Error: GA Optimized PD")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(11, 5))
plt.plot(time, best_result["u_cmd"], label="Commanded input")
plt.plot(time, best_result["u_delay"], label="Delayed input", linestyle="--")
plt.plot(time, best_result["u_applied"], label="Applied input", linewidth=2)
plt.xlabel("Time [s]")
plt.ylabel("Control Input")
plt.title("Control Input: Command, Delay, Rate Limit")
plt.legend()
plt.grid(True)
plt.show()